# Custom Chatbot Project

TODO: In this cell, write an explanation of which dataset you have chosen and why it is appropriate for this task

2023_fashion_trends.csv

I chose the included fashion trends dataset because it contains specific, structured information about 2023 styles that the model can reference. This lets it give more accurate, trend-aware fashion advice compared to a basic model that only gives general suggestions.

## Data Wrangling

TODO: In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [6]:
import pandas as pd
import os
import requests
import openai
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Setup
api_keys = "######"
openai.api_base = "https://openai.vocareum.com/v1"
openai.api_key = api_keys
openai_engine = "gpt-3.5-turbo-instruct"
embedding_model = "text-embedding-ada-002"

In [7]:
# The CSV file
df = pd.read_csv("data/2023_fashion_trends.csv")
df['text'] = df['Trends'].str.lower()

print("\nExample  :")
print(f"Original: {df['Trends'].iloc[0][:60]}...")
print(f"Lowercase: {df['text'].iloc[0][:60]}...")


Example  :
Original: 2023 Fashion Trend: Red. Glossy red hues took over the Fall ...
Lowercase: 2023 fashion trend: red. glossy red hues took over the fall ...


In [8]:
num_rows = len(df)
print("Number of rows:", num_rows)

# Confirming the DataFrame has ≥ 20 rows
if num_rows >= 20:
    print("we have 20 or more rows ")
else:
    print("Need more data - less than 20 rows")
    
def get_embedding(text):
    try:
        response = openai.Embedding.create(
            model=embedding_model,
            input=text
        )
        return np.array(response['data'][0]['embedding'])
    except Exception as e:
        return np.zeros(1536)

# Create embeddings for dataset
print("Creating embeddings...")
embeddings = []
for text in df['text']:
    embeddings.append(get_embedding(text))
embeddings = np.array(embeddings)
print(f"✓ Created {len(embeddings)} embeddings")

Number of rows: 82
we have 20 or more rows 
Creating embeddings...
✓ Created 82 embeddings


## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

In [13]:
prompt_template = """## ROLE
You are a fashion consultant specializing in 2023 trends.

## CONTEXT
The following are verified 2023 fashion trends from reputable sources:
{}

## TASK
Answer this fashion question using the context above.

## QUESTION
{}

## ANSWER
"""


def create_context_prompt(question, texts, embeddings, context_count=3):
    """Create prompt with template"""
    # Get question embedding
    q_embedding = get_embedding(question).reshape(1, -1)
    
    #Most similar texts
    similarities = cosine_similarity(q_embedding, embeddings)[0]
    top_indices = np.argsort(similarities)[-context_count:][::-1]
    
    # Build context
    context_parts = []
    for i, idx in enumerate(top_indices):
        trend = texts[idx]
        context_parts.append(f"TREND {i+1}: {trend}")
    
    context_text = "\n\n".join(context_parts)
    
    # template
    prompt = prompt_template.format(context_text, question)
    
    return prompt, top_indices

In [20]:
# Answer function
def answer_question(query, texts, embeddings, context_count=2):
    """Answer with 0 to N contexts"""
    
    if context_count == 0:
        # Basic query
        response = openai.Completion.create(
            engine=openai_engine,
            prompt=query,
            max_tokens=100,
            temperature=0.7
        )
        return response.choices[0].text.strip()
    
    else:
        # Custom query with embeddings and context
        prompt, _ = create_context_prompt(query, texts, embeddings, context_count)
        response = openai.Completion.create(
            engine=openai_engine,
            prompt=prompt,
            max_tokens=100,
            temperature=0.7
        )
        return response.choices[0].text.strip()

## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [21]:
print("\n" + "="*60)
print("CUSTOM PERFORMANCE DEMONSTRATION")
print("="*60)

texts = df['text'].tolist()


CUSTOM PERFORMANCE DEMONSTRATION


In [22]:
# Question 1
q1 = "What's good for spring daytime?"
print(f"\nQ1: {q1}")

print("\nBasic answer (no context):")
a1_basic = answer_question(q1, texts, embeddings, 0)
print(a1_basic)

print("\nCustom answer (1 context):")
a1_custom = answer_question(q1, texts, embeddings, 1)
print(a1_custom)




Q1: What's good for spring daytime?

Basic answer (no context):
1. Light Layers: As the weather starts to warm up, it's important to dress in light, breathable layers. Opt for items like a lightweight cardigan, a denim jacket, or a flowy scarf that you can easily take off if it gets too warm.

2. Floral Prints: Spring is the perfect time to bring out your favorite floral prints. Whether it's a dress, top, or even a pair of shoes, adding some floral to your outfit can instantly make it feel more

Custom answer (1 context):
According to the context, white cotton is a popular trend for spring daytime fashion in 2023. This includes slightly cropped embroidered blouses, loose-fitting shirts, and oversized tent dresses. Additionally, PVC ruffles, sequin suits, and innovative materials like Fabrica are also mentioned as popular choices for spring daytime fashion in 2023.


### Question 2

In [23]:
# Question 2
q2 = "Should I wear cargo pants?"
print(f"\n\nQ2: {q2}")

print("\nBasic answer (no context):")
a2_basic = answer_question(q2, texts, embeddings, 0)
print(a2_basic)

print("\nCustom answer (3 contexts):")
a2_custom = answer_question(q2, texts, embeddings, 3)
print(a2_custom)

print("\n" + "="*60)
print("SUMMARY: 2 questions, 4 total answers shown")
print("="*60)



Q2: Should I wear cargo pants?

Basic answer (no context):
It ultimately depends on your personal style and the occasion. Cargo pants can be practical and comfortable for outdoor activities or casual outings, but may not be appropriate for more formal or professional settings. Consider the fit and style of the pants, as well as the rest of your outfit, before deciding if cargo pants are the right choice for you.

Custom answer (3 contexts):
As a fashion consultant specializing in 2023 trends, my recommendation is to definitely incorporate cargo pants into your wardrobe. They are one of the top trends for 2023 and have been updated with tailored silhouettes, interesting pocket placements, and elevated fabrics. This is not your average shapeless cargo pant from the past. Additionally, cargo pants are a part of the utilitarian wear trend which is big for 2023. So, yes, you should definitely consider adding a pair of cargo pants

SUMMARY: 2 questions, 4 total answers shown
